In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. Carga de datos reales (usando el motor que detecta automáticamente el formato)
df_minable = pd.read_csv('./dataset/minable_banco_berka.csv', sep=None, engine='python')

# 2. Limpieza de columnas y prevención de Fuga de Datos (Target Leakage)
# Borramos 'has_card' (nuestra Y) y las variables de 'card_type' para que el modelo no haga trampa.
columnas_a_borrar = [
    'has_card',
    'district_id',
    'card_type_gold',
    'card_type_junior',
    'card_type_sin_tarjeta'
]
columnas_presentes = [col for col in columnas_a_borrar if col in df_minable.columns]

X = df_minable.drop(columns=columnas_presentes)
y = df_minable['has_card']

# 3. División Estratificada (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"📊 Datos reales divididos: {X_train.shape[0]} para Entrenamiento | {X_test.shape[0]} para Prueba")

# 4. Entrenamiento y Evaluación
models = {
    'Regresion Logistica': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost (Gradient)': GradientBoostingClassifier(random_state=42)
}

print("\n⚙️ Evaluando Modelos con datos reales de Banco Berka...")
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        'Modelo': name,
        'AUC-ROC': round(roc_auc_score(y_test, y_proba), 3),
        'F1-Score': round(f1_score(y_test, y_pred), 3),
        'Recall': round(recall_score(y_test, y_pred), 3),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 3)
    })

df_results = pd.DataFrame(results)
print("-" * 65)
print(df_results.to_string(index=False))
print("-" * 65)

# 5. Explicabilidad (Top 10 Variables)
importances = models['Random Forest'].feature_importances_
df_importance = pd.DataFrame({'Atributo': X.columns, 'Importancia': importances}).sort_values(by='Importancia', ascending=False)
print("\n📊 Top 10 Importancia de Variables (Random Forest):")
print(df_importance.head(10).to_string(index=False))

📊 Datos reales divididos: 3600 para Entrenamiento | 900 para Prueba

⚙️ Evaluando Modelos con datos reales de Banco Berka...
-----------------------------------------------------------------
             Modelo  AUC-ROC  F1-Score  Recall  Precision
Regresion Logistica    0.864     0.463   0.365      0.631
      Random Forest    0.862     0.365   0.258      0.622
 XGBoost (Gradient)    0.869     0.448   0.360      0.593
-----------------------------------------------------------------

📊 Top 10 Importancia de Variables (Random Forest):
         Atributo  Importancia
      avg_balance     0.133543
      max_balance     0.090088
         net_flow     0.075334
      std_balance     0.074273
 avg_trans_amount     0.056310
   total_deposits     0.047060
    deposit_ratio     0.042629
total_withdrawals     0.040128
       birth_year     0.037927
      trans_count     0.036807
